# 02 - Preprocessing
Goal: clean the ASEADOS-SDN-IoT data and split it into train and test sets.

In [2]:
import numpy as np
import pandas as pd

CSV_PATH = '../data/ASEADOS_SDN_IoT.csv'
df = pd.read_csv(CSV_PATH, low_memory=False)
df.columns = df.columns.str.strip()

print('Start:',df.shape)

Start: (457044, 84)


In [7]:
mask = df['Flow Duration'] < 0 
print(mask.head())

0    False
1    False
2    False
3    False
4    False
Name: Flow Duration, dtype: bool


In [10]:
print('Negative Flow Duration rows:',mask.sum())
print('Percent:',mask.sum() / len(df) * 100)

Negative Flow Duration rows: 2
Percent: 0.0004375946298387026


In [11]:
print(pd.crosstab(df['Protocol'], df['Label']))

Label     Benign   Bot   DDoS     Dos  Prob
Protocol                                   
0          42740  9090  51465   16075  7920
6         216512     0      0  111654     0
17          1545     0      0      43     0


Check B: Bot, DDoS, and probe flows all have Protocol = 0. Since "Protocol" is a shortcut, it will be removed.

In [5]:
id_cols = ["Flow ID", "Src IP", "Src Port","Dst IP", "Dst Port", "Timestamp"]
df.drop(columns=list).duplicated().sum()

np.int64(29993)

In [7]:
mask = df.drop(columns=id_cols).duplicated()
dup_rows = df[mask]
print(dup_rows['Label'].value_counts())

Label
Benign    27046
Dos        1265
DDoS       1195
Prob        354
Bot         133
Name: count, dtype: int64


Most of the repetitions occur during normal traffic. This makes sense, as IoT devices constantly send the same small messages. Removing them causes a slight decrease in the "benign" category, but that’s not an issue.

In [9]:
drop_cols = id_cols + ["Protocol"]
df = df.drop(columns=drop_cols)
print('After dropping ID columns + Protocol:', df.shape)

After dropping ID columns + Protocol: (457044, 77)


In [10]:
nunique = df.nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
df = df.drop(columns=constant_cols)
print('After dropping constant columns:',df.shape)

After dropping constant columns: (457044, 63)
